# Module 2 Homework (2026)

## Notes

- Q1 and Q2 use IPOscoop, as required by the homework.
- `StringIO` avoids problems when passing HTML text to `read_html`.
- Newer `yfinance` versions can return MultiIndex columns, so the download cell reshapes them before calculating returns.
- The live sources can change, so row counts may be slightly different.

In [9]:
# Uncomment in a fresh environment if needed:
# %pip install pandas numpy requests lxml yfinance pyarrow gdown

import io
import re
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import yfinance as yf

pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', lambda value: f'{value:,.4f}')

## Question 1: Withdrawn IPOs by company type

I first keep filings before September 11, 2026, then keep only withdrawn IPOs. After that I classify the company names and add up the estimated value for each company type. The largest value is the answer.

In [26]:
Q1_URL = 'https://www.iposcoop.com/ipos-recently-filed/'
response = requests.get(Q1_URL, timeout=30, headers={'User-Agent': 'Mozilla/5.0'})
response.raise_for_status()
q1_tables = pd.read_html(io.StringIO(response.text))
q1_raw = q1_tables[0].copy()
q1_raw.columns = [str(column).strip() for column in q1_raw.columns]

# Convert the filing date before filtering. The strict '<' cutoff excludes
# filings on September 11, 2026 and keeps only earlier filings.
q1_raw['File Date'] = pd.to_datetime(q1_raw['File Date'], errors='coerce')
q1 = q1_raw[q1_raw['File Date'] < pd.Timestamp('2026-09-11')].copy()

print('Rows before cutoff:', len(q1_raw))
print('Rows before September 11, 2026:', len(q1))
q1.head()

Rows before cutoff: 500
Rows before September 11, 2026: 463


,File Date,Company,Symbol,Managers,Shares (millions),Price Low,Price High,Est $ Vol (millions),Expected To Trade,SCOOP Rating
37,2026-09-10,Allarity Acquisition Corp.,ALLNU,Maxim Group,10.0000,$10.00,$10.00,$100.00,TBA,S/O
38,2026-09-10,Motive Technologies (Withdrawn),MTVE,J.P.Morgan/Citigroup/Barclays/Jefferies/RBC Ca...,0.0000,NaN,NaN,$100.00,Withdrawn,S/O
39,2026-09-09,Elevation Acquisition Group,ELEVU,Maxim Group,10.0000,$10.00,$10.00,$100.00,TBA,S/O
40,2026-09-09,Football Manager SPAC,FBLLU,D. Boral Capital,10.0000,$10.00,$10.00,$100.00,TBA,S/O
41,2026-09-09,Harbour Island Acquisition Corp. I,HRIAU,Clear Street,10.0000,$10.00,$10.00,$100.00,TBA,S/O


In [27]:
# Keep only withdrawn IPOs from the already date-filtered q1 DataFrame.
q1 = q1[q1['Expected To Trade'].eq('Withdrawn')].copy()
print('Withdrawn rows before September 11, 2026:', len(q1))

def classify_company(name):
    name = str(name)
    rules = [
        ('Technologies', 'Technologies'),
        ('Acquisition Corp', 'Acquisition Corp'),
        ('Acquisition Corporation', 'Acquisition Corp'),
        ('Corp', 'Acquisition Corp'),
        ('Inc', 'Inc.'),
        ('Incorporated', 'Inc.'),
        ('Group', 'Group'),
        ('Ltd', 'Limited'),
        ('Limited', 'Limited'),
        ('Holdings', 'Holdings'),
        ('Holding', 'Holdings'),
    ]
    for pattern, label in rules:
        if pattern in name:
            return label
    return 'Other'

q1['Company Type'] = q1['Company'].map(classify_company)

def parse_number(value):
    if pd.isna(value) or str(value).strip() == '-':
        return np.nan
    match = re.search(r'[-+]?[0-9]*\.?[0-9]+', str(value).replace(',', ''))
    return float(match.group()) if match else np.nan

q1['Price Low Parsed'] = q1['Price Low'].map(parse_number)
q1['Price High Parsed'] = q1['Price High'].map(parse_number)
q1['Avg_price'] = q1[['Price Low Parsed', 'Price High Parsed']].mean(axis=1)
q1['Shares (millions)'] = q1['Shares (millions)'].map(parse_number)
q1['Est $ Vol (millions)'] = q1['Est $ Vol (millions)'].map(parse_number)

# Use shares multiplied by the average price when available. Otherwise,
# fall back to the estimated offering volume.
q1['Shares_offered_value'] = q1['Shares (millions)'] * q1['Avg_price']
q1['Shares_offered_value'] = q1['Shares_offered_value'].fillna(q1['Est $ Vol (millions)'])

q1_summary = q1.groupby('Company Type')['Shares_offered_value'].sum().sort_values(ascending=False)
q1_summary

Withdrawn rows before September 11, 2026: 31


,Shares_offered_value
Company Type,
Acquisition Corp,499.9850
Inc.,351.0000
Holdings,311.6575
Other,290.4450
Limited,197.2500
Technologies,184.9000
Group,32.5000


## Question 2: Median Sharpe ratio for 2025 IPOs

I filter the 2025 IPOs by offer date, remove rows with zero return, download the available prices, and calculate the Sharpe ratio for September 11, 2026.

In [16]:
Q2_URL = 'https://www.iposcoop.com/2025-pricings/'
response = requests.get(
    Q2_URL,
    timeout=30,
    headers={'User-Agent': 'Mozilla/5.0'},
)
response.raise_for_status()

q2_raw = pd.read_html(io.StringIO(response.text))[0].copy()
q2_raw.columns = [
    re.sub(r'\s+', ' ', str(column)).strip()
    for column in q2_raw.columns
]

q2_raw['Offer Date'] = pd.to_datetime(
    q2_raw['Offer Date'],
    errors='coerce',
)
q2_raw['Return'] = pd.to_numeric(
    q2_raw['Return'].astype(str).str.replace('%', '', regex=False),
    errors='coerce',
)

q2 = q2_raw[
    (q2_raw['Offer Date'] < pd.Timestamp('2025-09-01'))
    & (q2_raw['Return'] != 0)
].copy()

# IPOscoop calls this column 'Symbol'. Normalize it to 'Ticker' for the
# yfinance and downstream analysis code.
column_lookup = {
    str(column).strip().lower(): column
    for column in q2.columns
}
symbol_column = next(
    (
        column_lookup[name]
        for name in ('ticker', 'symbol', 'company symbol')
        if name in column_lookup
    ),
    None,
)
if symbol_column is None:
    raise KeyError(f'Could not find a ticker column. Available columns: {list(q2.columns)}')

q2['Ticker'] = q2[symbol_column].astype('string').str.strip()

print('Symbol column used:', symbol_column)
print('Filtered IPO rows:', len(q2))
q2[['Ticker', 'Offer Date', 'Return']].head()

Symbol column used: Symbol
Filtered IPO rows: 146


,Ticker,Offer Date,Return
68,PMI,2025-08-29,-58.0000
69,TGHL,2025-08-28,-90.0000
70,THH,2025-08-28,636.7500
71,CURX,2025-08-26,-90.7500
72,CEPF,2025-08-21,1.2000


In [28]:
# Download in batches. Missing or delisted tickers are expected.
tickers = q2['Ticker'].dropna().unique().tolist()


def download_price_batch(batch):
    batch_prices = yf.download(
        batch,
        period='max',
        auto_adjust=False,
        progress=False,
        group_by='ticker',
    )

    if isinstance(batch_prices.columns, pd.MultiIndex):
        # Convert columns such as (AAPL, Close) into one row per date/ticker.
        batch_prices = (
            batch_prices
            .stack(level=0, dropna=False)
            .rename_axis(index=['date', 'Ticker'])
            .reset_index()
        )
    else:
        # yfinance may return ordinary columns for a one-ticker request.
        batch_prices = batch_prices.reset_index()
        batch_prices['Ticker'] = batch[0]

    return batch_prices


batch_size = 50
price_frames = [
    download_price_batch(tickers[start:start + batch_size])
    for start in range(0, len(tickers), batch_size)
]
stocks_df = pd.concat(price_frames, ignore_index=True)
stocks_df['date'] = pd.to_datetime(stocks_df['date']).dt.normalize()
stocks_df = stocks_df.dropna(subset=['Close']).sort_values(['Ticker', 'date'])

# Growth after approximately one trading year.
stocks_df['growth_252d'] = (
    stocks_df['Close']
    / stocks_df.groupby('Ticker')['Close'].shift(252)
)

# Annualized 30-trading-day rolling volatility.
stocks_df['volatility'] = stocks_df.groupby('Ticker')['Close'].transform(
    lambda prices: prices.rolling(30).std() * np.sqrt(252)
)

# The assignment uses a 5% risk-free rate.
stocks_df['Sharpe'] = (
    stocks_df['growth_252d'] - 0.05
) / stocks_df['volatility']

q2_result = stocks_df[
    stocks_df['date'].eq(pd.Timestamp('2026-09-11'))
][['growth_252d', 'volatility', 'Sharpe']].describe()
q2_result

ERROR:yfinance:
3 Failed downloads:
ERROR:yfinance:['EMPG', 'CAEP', 'MJID']: YFTzMissingError('possibly delisted; no timezone found')
/tmp/ipykernel_1403/2767858434.py:18: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  .stack(level=0, dropna=False)
ERROR:yfinance:
4 Failed downloads:
ERROR:yfinance:['SDM', 'CEPT', 'AHL', 'PTNM']: YFTzMissingError('possibly delisted; no timezone found')
/tmp/ipykernel_1403/2767858434.py:18: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  .stack(level=0, dropna=False)
ERROR:yfinance:
7 Failed downloads:
ERROR:yfinance:['MCTR', 'AGH', 'TBH', 

Price,growth_252d,volatility,Sharpe
count,130.0000,131.0000,130.0000
mean,1.0521,27.5440,NaN
std,3.0492,66.8789,NaN
min,0.0010,0.0000,-inf
25%,0.1410,2.2652,0.0120
50%,0.5945,7.8857,0.0490
75%,1.0388,23.9685,0.1336
max,33.6383,627.6981,inf


## Question 3: Fixed-month holding strategy

For each IPO, I compare the first closing price with prices 21, 42, ..., 252 trading days later. Then I compare the medians to find the best holding period.

In [21]:
# Calculate future growth for 1 to 12 months, using 21 trading days per month.
holding_days = {
    f'future_growth_{month}_m': 21 * month
    for month in range(1, 13)
}

stocks_df = stocks_df.sort_values(['Ticker', 'date']).copy()
for column, days in holding_days.items():
    stocks_df[column] = (
        stocks_df.groupby('Ticker')['Close'].shift(-days)
        / stocks_df['Close']
    )

# Keep the first available trading day for each ticker.
first_days = (
    stocks_df.sort_values(['Ticker', 'date'])
    .groupby('Ticker', as_index=False)
    .first()
)

growth_columns = list(holding_days)
q3_summary = first_days[growth_columns].describe().T
q3_summary['median'] = q3_summary['50%']
q3_summary = q3_summary.sort_values('median', ascending=False)
q3_summary

best_column = q3_summary.index[0]
best_month = int(best_column.split('_')[2])
best_median_growth = q3_summary.iloc[0]['median']
print(f'Best holding period: {best_month} month(s)')
print(f'Maximum median growth: {best_median_growth:.4f}')

Best holding period: 1 month(s)
Maximum median growth: 0.9354


## Question 4: RSI-based trading strategy

I use the provided data, keep RSI values below 30 between 2000 and June 1, 2025, and calculate the result of investing $1,000 in every signal.

In [24]:
# Download the provided dataset only when it is not already present.
import gdown

file_id = '1grCTCzMZKY5sJRtdbLVCXg8JXA8VPyg-'
data_path = Path('data.parquet')
if not data_path.exists():
    gdown.download(
        f'https://drive.google.com/uc?id={file_id}',
        str(data_path),
        quiet=False,
    )

# Load the precomputed technical and macro indicators.
df = pd.read_parquet(data_path, engine='pyarrow')
df.columns = [str(column).strip() for column in df.columns]

# Normalize the date column used by the assignment.
date_column = next(
    (column for column in ('date', 'Date', 'datetime', 'Datetime') if column in df.columns),
    None,
)
if date_column is None:
    raise KeyError(f'Could not find a date column. Available columns: {list(df.columns)}')
if date_column != 'date':
    df = df.rename(columns={date_column: 'date'})
df['date'] = pd.to_datetime(df['date'], errors='coerce')

# The downloaded file may name RSI as rsi, RSI, or rsi_14.
rsi_column = next(
    (
        column for column in df.columns
        if str(column).strip().lower() in {'rsi', 'rsi_14', 'rsi14'}
    ),
    None,
)
if rsi_column is None:
    raise KeyError(f'Could not find an RSI column. Available columns: {list(df.columns)}')
df['RSI'] = pd.to_numeric(df[rsi_column], errors='coerce')

required_columns = {'RSI', 'growth_future_30d', 'date'}
missing_columns = required_columns.difference(df.columns)
if missing_columns:
    raise KeyError(f'Missing required columns: {sorted(missing_columns)}')

# Select oversold signals in the assignment's date range.
selected_df = df[
    df['RSI'].lt(30)
    & df['date'].between('2000-01-01', '2025-06-01')
].copy()
selected_df = selected_df.dropna(subset=['growth_future_30d'])

# Invest $1,000 for every signal and use the 30-day forward growth factor.
selected_df['trade_return'] = selected_df['growth_future_30d'] - 1
net_income = 1000 * selected_df['trade_return'].sum()
net_income_thousands = net_income / 1000

print('RSI column used:', rsi_column)
print('Valid oversold trades:', len(selected_df))
print(f'Net income: ${net_income:,.2f}')
print(f'Net income ($ thousands): {net_income_thousands:,.2f}')
net_income_thousands

RSI column used: rsi
Valid oversold trades: 5206
Net income: $65,805.59
Net income ($ thousands): 65.81


np.float64(65.80558960747933)

**Q4 answer:** The strategy produces **$65.81 thousand** in net income from 5,206 valid RSI < 30 signals. This assumes a separate $1,000 investment per signal and ignores fees, slippage, and overlapping capital constraints.

## Question 5: Optional exploration

### How would you change the strategy to increase profitability?

I would trade fewer, more liquid stocks and use recent momentum, volatility, and market-relative performance to choose signals. I would include fees and position limits, and test the rules on later data using only information known before each trade. This avoids look-ahead bias.

The example below uses positive previous 21-day growth and below-median volatility. It improved the median return and win rate, but the mean return was still negative, so it needs more testing before being considered profitable.

In [29]:
# The filter uses only information available before each trading date.
q5_df = stocks_df.sort_values(['Ticker', 'date']).copy()
q5_df['past_growth_21d'] = (
    q5_df['Close']
    / q5_df.groupby('Ticker')['Close'].shift(21)
    - 1
)
q5_df['volatility_30d'] = q5_df.groupby('Ticker')['Close'].transform(
    lambda prices: prices.pct_change().rolling(30).std() * np.sqrt(252)
)
q5_df['future_growth_21d'] = (
    q5_df.groupby('Ticker')['Close'].shift(-21)
    / q5_df['Close']
)

q5_df = q5_df.dropna(
    subset=['past_growth_21d', 'volatility_30d', 'future_growth_21d']
).copy()

# Use an earlier period to choose thresholds, then evaluate only later data.
q5_split_date = pd.Timestamp('2025-09-01')
q5_train = q5_df[q5_df['date'] < q5_split_date]
q5_test = q5_df[q5_df['date'] >= q5_split_date].copy()

past_growth_threshold = q5_train['past_growth_21d'].median()
volatility_threshold = q5_train['volatility_30d'].median()

q5_test['filtered_signal'] = (
    (q5_test['past_growth_21d'] > past_growth_threshold)
    & (q5_test['volatility_30d'] < volatility_threshold)
)

q5_baseline = q5_test['future_growth_21d'] - 1
q5_filtered = q5_test.loc[q5_test['filtered_signal'], 'future_growth_21d'] - 1

q5_results = pd.DataFrame({
    'strategy': ['Baseline: all dates', 'Filtered: positive trend + lower volatility'],
    'trades': [q5_baseline.count(), q5_filtered.count()],
    'mean_return': [q5_baseline.mean(), q5_filtered.mean()],
    'median_return': [q5_baseline.median(), q5_filtered.median()],
    'win_rate': [(q5_baseline > 0).mean(), (q5_filtered > 0).mean()],
})

print(f'Training cutoff: {q5_split_date.date()}')
print(f'Past-growth threshold: {past_growth_threshold:.4f}')
print(f'Volatility threshold: {volatility_threshold:.4f}')
q5_results

Training cutoff: 2025-09-01
Past-growth threshold: 0.0000
Volatility threshold: 0.7353


,strategy,trades,mean_return,median_return,win_rate
0,Baseline: all dates,31994,0.0004,-0.0359,0.3939
1,Filtered: positive trend + lower volatility,4912,-0.0182,-0.0077,0.4558
